<div dir="rtl" style="text-align: right;">

[![باز کردن در Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alireza-Akhavan/ThinkPython/blob/main/chapters_fa/chap13_fa.ipynb)

</div>

<div dir="rtl" style="text-align: right;">

# فصل ۱۳: فایل‌ها و پایگاه‌های داده

ترجمه‌ی فارسی فصل سیزدهم کتاب «Think Python» (ویرایش سوم) نوشته‌ی Allen B. Downey. متن اصلی و کدها از [وب‌سایت کتاب](https://allendowney.github.io/ThinkPython/chap13.html) و [مخزن GitHub](https://github.com/AllenDowney/ThinkPython/tree/v3/chapters) گرفته شده‌اند. متن کتاب با مجوز [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) و کدها با مجوز MIT منتشر شده‌اند؛ این ترجمه نیز با همان مجوز و فقط برای استفاده‌ی غیرتجاری منتشر می‌شود.

**چند نکته درباره‌ی این ترجمه:** (۱) در کتاب اصلی، نام کاربری در مسیر پوشه‌ی خانگی نمونه به یک شخصیت طنز اشاره دارد؛ من آن را با نام ساده‌ی <code dir="ltr">student</code> جایگزین کرده‌ام. (۲) عکس‌های این فصل از [Lorem Picsum](https://picsum.photos/)، سرویسی برای عکس‌های جانگهدار، گرفته شده‌اند و در فایل <code dir="ltr">photos.zip</code> کتاب هستند. (۳) خروجی‌های نمایش‌داده‌شده‌ی زیر کدها را با اجرای همین کدها روی فایل‌های کتاب به دست آورده‌ام. ترتیب فهرست فایل‌ها و نام فایل‌هایی که <code dir="ltr">shelve</code> می‌سازد به سیستم‌عامل و پیاده‌سازی <code dir="ltr">dbm</code> بستگی دارد؛ این موارد را در متن توضیح داده‌ام.

سلول‌های زیر در نوت‌بوک اصلی فایل‌های مثال این فصل را آماده می‌کنند:

</div>

<div dir="rtl" style="text-align: right;">

**ابتدا این سلول را اجرا کنید.** فایل‌های کمکی کتاب (مثل <code dir="ltr">thinkpython.py</code>) را دانلود می‌کند و دستورهای ویژه‌ی نوت‌بوک (مثل <code dir="ltr">%%add_method_to</code> و <code dir="ltr">%%expect</code>) را فعال می‌کند. برای اجرا به اینترنت نیاز دارید.

</div>

In [ ]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://raw.githubusercontent.com/Alireza-Akhavan/ThinkPython/main/thinkpython.py');
download('https://raw.githubusercontent.com/Alireza-Akhavan/ThinkPython/main/diagram.py');

import thinkpython

In [ ]:
# This cell downloads an archive file that contains the the files we'll
# use for the examples in this chapter.

download('https://raw.githubusercontent.com/Alireza-Akhavan/ThinkPython/main/photos.zip');

In [ ]:
# WARNING: This cell removes the photos/ directory if it already exists.
# Any files already in the photos/ directory will be deleted.

# !rm -rf photos/

In [ ]:
!unzip -o photos.zip

<div dir="rtl" style="text-align: right;">

بیشتر برنامه‌هایی که تا الان دیده‌ایم **زودگذر** (ephemeral) هستند، به این معنا که مدت کوتاهی اجرا می‌شوند و خروجی تولید می‌کنند، اما وقتی تمام می‌شوند، داده‌هایشان از بین می‌رود. هر بار که یک برنامه‌ی زودگذر را اجرا می‌کنید، از صفر شروع می‌شود.

برنامه‌های دیگر **پایدار** (persistent) هستند: مدت طولانی (یا همیشه) اجرا می‌شوند؛ دست‌کم بخشی از داده‌هایشان را در ذخیره‌سازی بلندمدت نگه می‌دارند؛ و اگر خاموش و دوباره راه‌اندازی شوند، از همان جایی که بودند ادامه می‌دهند.

یک راه ساده برای اینکه برنامه‌ها داده‌هایشان را نگه دارند خواندن و نوشتن فایل‌های متنی است. جایگزین همه‌کاره‌تر، ذخیره‌ی داده در یک پایگاه داده است. پایگاه‌های داده فایل‌های تخصصی‌ای هستند که کارآمدتر از فایل‌های متنی خوانده و نوشته می‌شوند و قابلیت‌های اضافی دارند.

در این فصل برنامه‌هایی می‌نویسیم که فایل‌های متنی و پایگاه‌های داده را می‌خوانند و می‌نویسند، و به‌عنوان تمرین برنامه‌ای می‌نویسید که مجموعه‌ای از عکس‌ها را برای پیدا کردن تکراری‌ها جست‌وجو می‌کند. اما قبل از اینکه با یک فایل کار کنید، باید آن را پیدا کنید، پس با نام فایل‌ها، مسیرها و دایرکتوری‌ها شروع می‌کنیم.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۳.۱. نام فایل‌ها و مسیرها

فایل‌ها در **دایرکتوری‌ها** (directories) که «پوشه» (folder) هم نامیده می‌شوند سازمان‌دهی می‌شوند. هر برنامه‌ی در حال اجرا یک **دایرکتوری کاری فعلی** (current working directory) دارد که دایرکتوری پیش‌فرض بیشتر عملیات است. برای مثال، وقتی یک فایل را باز می‌کنید، پایتون آن را در دایرکتوری کاری فعلی جست‌وجو می‌کند.

ماژول <code dir="ltr">os</code> توابعی برای کار با فایل‌ها و دایرکتوری‌ها دارد («os» مخفف «operating system»، یعنی سیستم‌عامل است). تابعی به نام <code dir="ltr">getcwd</code> دارد که نام دایرکتوری کاری فعلی را می‌گیرد.

</div>

In [ ]:
# this cell replaces `os.cwd` with a function that returns a fake path

import os

_real_getcwd = os.getcwd

def getcwd():
    return "/home/student"

os.getcwd = getcwd

In [ ]:
import os

os.getcwd()

<div dir="rtl" style="text-align: right;">

نتیجه در این مثال دایرکتوری خانگی کاربری به نام <code dir="ltr">student</code> است. رشته‌ای مثل <code dir="ltr">'/home/student'</code> که یک فایل یا دایرکتوری را مشخص می‌کند **مسیر** (path) نامیده می‌شود.

نام ساده‌ی فایل مثل <code dir="ltr">'memo.txt'</code> هم یک مسیر به حساب می‌آید، اما **مسیر نسبی** (relative path) است چون نام فایل را نسبت به دایرکتوری فعلی مشخص می‌کند. در این مثال، دایرکتوری فعلی <code dir="ltr">/home/student</code> است، پس <code dir="ltr">'memo.txt'</code> معادل مسیر کامل <code dir="ltr">'/home/student/memo.txt'</code> است.

مسیری که با <code dir="ltr">/</code> شروع می‌شود به دایرکتوری فعلی بستگی ندارد – **مسیر مطلق** (absolute path) نامیده می‌شود. برای پیدا کردن مسیر مطلق یک فایل، می‌توانید از <code dir="ltr">abspath</code> استفاده کنید.

</div>

In [ ]:
os.path.abspath('memo.txt')

<div dir="rtl" style="text-align: right;">

مسیر ساختگی <code dir="ltr">/home/student</code> فقط برای نمایش بود. این سلول تابع واقعی <code dir="ltr">os.getcwd</code> را برمی‌گرداند، چون بدون آن <code dir="ltr">shelve</code> در پایتون ۳٫۱۳ نمی‌تواند فایل پایگاه داده را باز کند. (این سلول در کتاب اصلی نیست.)

</div>

In [ ]:
os.getcwd = _real_getcwd

<div dir="rtl" style="text-align: right;">

ماژول <code dir="ltr">os</code> توابع دیگری برای کار با نام فایل‌ها و مسیرها دارد. <code dir="ltr">listdir</code> فهرستی از محتویات دایرکتوری داده‌شده را، شامل فایل‌ها و دایرکتوری‌های دیگر، برمی‌گرداند. این مثالی است که محتویات دایرکتوری‌ای به نام <code dir="ltr">photos</code> را فهرست می‌کند.

</div>

In [ ]:
os.listdir('photos')

<div dir="rtl" style="text-align: right;">

(ترتیب عنصرها به سیستم‌عامل بستگی دارد و ممکن است در رایانه‌ی شما فرق کند.)

این دایرکتوری یک فایل متنی به نام <code dir="ltr">notes.txt</code> و سه دایرکتوری دارد. دایرکتوری‌ها فایل‌های تصویری با قالب JPEG دارند.

</div>

In [ ]:
os.listdir('photos/jan-2023')

<div dir="rtl" style="text-align: right;">

برای بررسی اینکه فایل یا دایرکتوری وجود دارد یا نه، می‌توانیم از <code dir="ltr">os.path.exists</code> استفاده کنیم.

</div>

In [ ]:
os.path.exists('photos')

In [ ]:
os.path.exists('photos/apr-2023')

<div dir="rtl" style="text-align: right;">

برای بررسی اینکه مسیر به یک فایل اشاره می‌کند یا دایرکتوری، می‌توانیم از <code dir="ltr">isdir</code> استفاده کنیم که اگر مسیر به دایرکتوری اشاره کند <code dir="ltr">True</code> برمی‌گرداند.

</div>

In [ ]:
os.path.isdir('photos')

<div dir="rtl" style="text-align: right;">

و <code dir="ltr">isfile</code> که اگر مسیر به فایل اشاره کند <code dir="ltr">True</code> برمی‌گرداند.

</div>

In [ ]:
os.path.isfile('photos/notes.txt')

<div dir="rtl" style="text-align: right;">

یک چالش کار با مسیرها این است که در سیستم‌عامل‌های مختلف شکل متفاوتی دارند. در macOS و سیستم‌های یونیکس مثل لینوکس، نام دایرکتوری‌ها و فایل‌های یک مسیر با اسلش رو به جلو، <code dir="ltr">/</code>، از هم جدا می‌شوند. ویندوز از اسلش رو به عقب، <code dir="ltr">\</code>، استفاده می‌کند. پس اگر این مثال‌ها را در ویندوز اجرا کنید، در مسیرها اسلش رو به عقب می‌بینید و باید اسلش‌های رو به جلوی مثال‌ها را جایگزین کنید.

یا، برای نوشتن کدی که روی هر دو سیستم کار کند، می‌توانید از <code dir="ltr">os.path.join</code> استفاده کنید که نام دایرکتوری‌ها و فایل‌ها را بسته به سیستم‌عاملی که استفاده می‌کنید با اسلش رو به جلو یا رو به عقب به یک مسیر تبدیل می‌کند.

</div>

In [ ]:
os.path.join('photos', 'jan-2023', 'photo1.jpg')

<div dir="rtl" style="text-align: right;">

در ادامه‌ی این فصل از این توابع برای جست‌وجوی مجموعه‌ای از دایرکتوری‌ها و پیدا کردن همه‌ی فایل‌های تصویری استفاده می‌کنیم.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۳.۲. f-string ها

یک راه برای اینکه برنامه‌ها داده‌ها را ذخیره کنند نوشتن آن‌ها در یک فایل متنی است. برای مثال، فرض کنید شما یک «شتردیده‌بان» هستید و می‌خواهید تعداد شترهایی را که در یک دوره‌ی مشاهده دیده‌اید ثبت کنید. و فرض کنید در یک سال و نیم <code dir="ltr">23</code> شتر دیده‌اید. داده‌ها در دفتر شترشماری شما ممکن است این‌طور باشد.

</div>

In [ ]:
num_years = 1.5
num_camels = 23

<div dir="rtl" style="text-align: right;">

برای نوشتن این داده‌ها در یک فایل، می‌توانید از متد <code dir="ltr">write</code> که در فصل ۸ دیدیم استفاده کنید. آرگومان <code dir="ltr">write</code> باید رشته باشد، پس اگر بخواهیم مقدارهای دیگر را در فایل بگذاریم، باید آن‌ها را به رشته تبدیل کنیم. آسان‌ترین راه برای این کار تابع داخلی <code dir="ltr">str</code> است.

این‌طور به نظر می‌رسد:

</div>

In [ ]:
writer = open('camel-spotting-book.txt', 'w')
writer.write(str(num_years))
writer.write(str(num_camels))
writer.close()

<div dir="rtl" style="text-align: right;">

این کار می‌کند، اما <code dir="ltr">write</code> فاصله یا خط جدید اضافه نمی‌کند مگر اینکه صریحاً بگذارید. اگر فایل را دوباره بخوانیم، می‌بینیم دو عدد به هم چسبیده‌اند.

</div>

In [ ]:
open('camel-spotting-book.txt').read()

<div dir="rtl" style="text-align: right;">

دست‌کم باید بین عددها فاصله بگذاریم. و حالا که این کار را می‌کنیم، متن توضیحی هم اضافه کنیم.

برای نوشتن ترکیبی از رشته‌ها و مقدارهای دیگر، می‌توانیم از **f-string** استفاده کنیم، که رشته‌ای است که حرف <code dir="ltr">f</code> قبل از علامت نقل‌قول بازکننده دارد و یک یا چند عبارت پایتون در آکولاد دارد. f-string زیر یک عبارت دارد، که یک نام متغیر است.

</div>

In [ ]:
f'I have spotted {num_camels} camels'

<div dir="rtl" style="text-align: right;">

نتیجه رشته‌ای است که در آن عبارت ارزیابی شده و با نتیجه جایگزین شده است. می‌تواند بیش از یک عبارت وجود داشته باشد.

</div>

In [ ]:
f'In {num_years} years I have spotted {num_camels} camels'

<div dir="rtl" style="text-align: right;">

و عبارت‌ها می‌توانند عملگر و فراخوانی تابع داشته باشند.

</div>

In [ ]:
line = f'In {round(num_years * 12)} months I have spotted {num_camels} camels'
line

<div dir="rtl" style="text-align: right;">

پس می‌توانیم داده‌ها را این‌طور در یک فایل متنی بنویسیم.

</div>

In [ ]:
writer = open('camel-spotting-book.txt', 'w')
writer.write(f'Years of observation: {num_years}\n')
writer.write(f'Camels spotted: {num_camels}\n')
writer.close()

<div dir="rtl" style="text-align: right;">

هر دو f-string به دنباله‌ی <code dir="ltr">\n</code> ختم می‌شوند که یک نویسه‌ی خط جدید اضافه می‌کند.

می‌توانیم فایل را این‌طور دوباره بخوانیم:

</div>

In [ ]:
data = open('camel-spotting-book.txt').read()
print(data)

<div dir="rtl" style="text-align: right;">

در f-string، عبارت داخل آکولاد به رشته تبدیل می‌شود، پس می‌توانید فهرست، دیکشنری و نوع‌های دیگر را هم بگنجانید.

</div>

In [ ]:
t = [1, 2, 3]
d = {'one': 1}
f'Here is a list {t} and a dictionary {d}'

<div dir="rtl" style="text-align: right;">

اگر f-string عبارت نامعتبری داشته باشد، نتیجه خطاست.

</div>

In [ ]:
%%expect TypeError

f'This is not a valid expression {t + 2}'

<div dir="rtl" style="text-align: right;">

## ۱۳.۳. YAML

یکی از دلایل خواندن و نوشتن فایل در برنامه‌ها ذخیره‌ی **داده‌های پیکربندی** (configuration data) است، یعنی اطلاعاتی که مشخص می‌کند برنامه چه کاری و چگونه باید انجام دهد.

برای مثال، در برنامه‌ای که عکس‌های تکراری را جست‌وجو می‌کند، ممکن است دیکشنری‌ای به نام <code dir="ltr">config</code> داشته باشیم که نام دایرکتوری‌ای که باید جست‌وجو شود، نام دایرکتوری دیگری که باید نتیجه‌ها را در آن ذخیره کند و فهرستی از پسوندهای فایلی که باید برای شناسایی فایل‌های تصویری به کار ببرد را در بر دارد.

ممکن است این‌طور باشد:

</div>

In [ ]:
config = {
    'photo_dir': 'photos',
    'data_dir': 'photo_info',
    'extensions': ['jpg', 'jpeg'],
}

<div dir="rtl" style="text-align: right;">

برای نوشتن این داده‌ها در یک فایل متنی، می‌توانیم مثل بخش قبل از f-string ها استفاده کنیم. اما آسان‌تر است که از ماژولی به نام <code dir="ltr">yaml</code> استفاده کنیم که دقیقاً برای همین کار طراحی شده است.

ماژول <code dir="ltr">yaml</code> توابعی برای کار با فایل‌های YAML فراهم می‌کند، که فایل‌های متنی‌ای هستند که طوری قالب‌بندی شده‌اند که هم برای انسان‌ها *و* هم برای برنامه‌ها خواندن و نوشتنشان آسان باشد.

این مثالی است که از تابع <code dir="ltr">dump</code> برای نوشتن دیکشنری <code dir="ltr">config</code> در یک فایل YAML استفاده می‌کند.

</div>

In [ ]:
# this cell installs the pyyaml package, which provides the yaml module

try:
    import yaml
except ImportError:
    !pip install pyyaml

In [ ]:
import yaml

config_filename = 'config.yaml'
writer = open(config_filename, 'w')
yaml.dump(config, writer)
writer.close()

<div dir="rtl" style="text-align: right;">

اگر محتویات فایل را دوباره بخوانیم، می‌توانیم ببینیم قالب YAML چه شکلی است.

</div>

In [ ]:
readback = open(config_filename).read()
print(readback)

<div dir="rtl" style="text-align: right;">

حالا می‌توانیم از <code dir="ltr">safe_load</code> برای خواندن دوباره‌ی فایل YAML استفاده کنیم.

</div>

In [ ]:
reader = open(config_filename)
config_readback = yaml.safe_load(reader)
config_readback

<div dir="rtl" style="text-align: right;">

نتیجه دیکشنری جدیدی است که اطلاعات یکسانی با دیکشنری اصلی دارد، اما همان دیکشنری نیست.

</div>

In [ ]:
config is config_readback

<div dir="rtl" style="text-align: right;">

تبدیل یک شیء مثل دیکشنری به رشته **سریال‌سازی** (serialization) نامیده می‌شود. تبدیل رشته به شیء **سریال‌زدایی** (deserialization) نام دارد. اگر یک شیء را سریال کنید و بعد سریال‌زدایی کنید، نتیجه باید هم‌ارز با شیء اصلی باشد.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۳.۴. Shelve

تا اینجا فایل‌های متنی را می‌خواندیم و می‌نوشتیم – حالا پایگاه‌های داده را بررسی کنیم. **پایگاه داده** (database) فایلی است که برای ذخیره‌ی داده سازمان‌دهی شده است. بعضی پایگاه‌های داده مثل جدولی با سطرها و ستون‌های اطلاعات سازمان‌دهی شده‌اند. بعضی دیگر مثل دیکشنری هستند که کلیدها را به مقدارها نگاشت می‌کنند؛ این‌ها گاهی **ذخیره‌گاه کلید-مقدار** (key-value store) نامیده می‌شوند.

ماژول <code dir="ltr">shelve</code> توابعی برای ساختن و به‌روزرسانی یک ذخیره‌گاه کلید-مقدار به نام «قفسه» (shelf) فراهم می‌کند. به‌عنوان مثال، قفسه‌ای می‌سازیم که توضیح‌های (caption) تصویرهای دایرکتوری <code dir="ltr">photos</code> را نگه دارد. از دیکشنری <code dir="ltr">config</code> برای گرفتن نام دایرکتوری‌ای که باید قفسه را در آن بگذاریم استفاده می‌کنیم.

</div>

In [ ]:
config['data_dir']

<div dir="rtl" style="text-align: right;">

اگر این دایرکتوری از قبل وجود نداشته باشد، می‌توانیم با <code dir="ltr">os.makedirs</code> آن را بسازیم.

</div>

In [ ]:
os.makedirs(config['data_dir'], exist_ok=True)

<div dir="rtl" style="text-align: right;">

و از <code dir="ltr">os.path.join</code> برای ساختن مسیری که نام دایرکتوری و نام فایل قفسه، یعنی <code dir="ltr">captions</code>، را دارد.

</div>

In [ ]:
db_file = os.path.join(config['data_dir'], 'captions')
db_file

<div dir="rtl" style="text-align: right;">

حالا می‌توانیم از <code dir="ltr">shelve.open</code> برای باز کردن فایل قفسه استفاده کنیم. آرگومان <code dir="ltr">c</code> نشان می‌دهد که اگر لازم است فایل باید ساخته شود.

</div>

In [ ]:
import shelve

db = shelve.open(db_file, 'c')
db

<div dir="rtl" style="text-align: right;">

اگر خطایی مثل <code dir="ltr">db type could not be determined</code> گرفتید، محتمل‌ترین علت این است که فایلی با همین نام از قبل وجود دارد اما پایگاه داده‌ی <code dir="ltr">shelve</code> معتبری نیست (مثلاً ممکن است خراب شده باشد یا چیز دیگری آن را ساخته باشد).

در آن صورت، ساده‌ترین راه‌حل حذف فایل موجود و اجرای دوباره‌ی کد است تا <code dir="ltr">shelve.open</code> بتواند پایگاه داده‌ی جدیدی بسازد.

مقدار بازگشتی رسماً یک شیء <code dir="ltr">DbfilenameShelf</code> است که غیررسمی‌تر شیء قفسه نامیده می‌شود.

شیء قفسه از جهات زیادی مثل دیکشنری رفتار می‌کند. برای مثال، می‌توانیم از عملگر کروشه برای اضافه کردن یک مورد، که نگاشتی از یک کلید به یک مقدار است، استفاده کنیم.

</div>

In [ ]:
key = 'jan-2023/photo1.jpg' 
db[key] = 'Cat nose'

<div dir="rtl" style="text-align: right;">

در این مثال، کلید مسیر یک فایل تصویری و مقدار رشته‌ای است که تصویر را توصیف می‌کند.

همچنین از عملگر کروشه برای جست‌وجوی یک کلید و گرفتن مقدار متناظر استفاده می‌کنیم.

</div>

In [ ]:
value = db[key]
value

<div dir="rtl" style="text-align: right;">

اگر نسبت‌دهی دیگری به یک کلید موجود انجام دهید، <code dir="ltr">shelve</code> مقدار قدیمی را جایگزین می‌کند.

</div>

In [ ]:
db[key] = 'Close up view of a cat nose'
db[key]

<div dir="rtl" style="text-align: right;">

بعضی متدهای دیکشنری، مثل <code dir="ltr">keys</code>، <code dir="ltr">values</code> و <code dir="ltr">items</code>، روی شیءهای قفسه هم کار می‌کنند.

</div>

In [ ]:
list(db.keys())

In [ ]:
list(db.values())

<div dir="rtl" style="text-align: right;">

می‌توانیم از عملگر <code dir="ltr">in</code> برای بررسی اینکه کلیدی در قفسه هست یا نه استفاده کنیم.

</div>

In [ ]:
key in db

<div dir="rtl" style="text-align: right;">

و می‌توانیم از دستور <code dir="ltr">for</code> برای پیمایش کلیدها استفاده کنیم.

</div>

In [ ]:
for key in db:
    print(key, ':', db[key])

<div dir="rtl" style="text-align: right;">

مثل فایل‌های دیگر، وقتی کارتان تمام شد باید پایگاه داده را ببندید.

</div>

In [ ]:
db.close()

<div dir="rtl" style="text-align: right;">

حالا اگر محتویات دایرکتوری داده را فهرست کنیم، دو فایل می‌بینیم.

</div>

In [ ]:
# When you open a shelve file, a backup file is created that has the suffix `.bak`.
# If you run this notebook more than once, you might see that file left behind.
# This cell removes it so the output shown in the book is correct.

!rm -f photo_info/captions.bak

In [ ]:
os.listdir(config['data_dir'])

<div dir="rtl" style="text-align: right;">

(این خروجی همان است که در کتاب آمده. نام و تعداد فایل‌هایی که <code dir="ltr">shelve</code> می‌سازد به پیاده‌سازی <code dir="ltr">dbm</code> در سیستم شما بستگی دارد؛ روی بعضی سیستم‌ها فقط یک فایل با نام <code dir="ltr">captions</code> می‌بینید، که در اجرای من این‌طور بود.)

<code dir="ltr">captions.dat</code> داده‌ای را که همین الان ذخیره کردیم دارد. <code dir="ltr">captions.dir</code> اطلاعاتی درباره‌ی سازمان‌دهی پایگاه داده دارد که دسترسی به آن را کارآمدتر می‌کند. پسوند <code dir="ltr">dir</code> مخفف «directory» است، اما ربطی به دایرکتوری‌هایی که فایل‌ها را در خود دارند و تا الان با آن‌ها کار می‌کردیم ندارد.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۳.۵. ذخیره‌ی ساختارهای داده

در مثال قبل، کلیدها و مقدارهای قفسه رشته بودند. اما می‌توانیم از قفسه برای نگه داشتن ساختارهای داده مثل فهرست‌ها و دیکشنری‌ها هم استفاده کنیم.

به‌عنوان مثال، مثال آناگرام را از یکی از تمرین‌های فصل ۱۱ دوباره بررسی می‌کنیم. به یاد بیاورید که دیکشنری‌ای ساختیم که رشته‌ی مرتب‌شده‌ی حرف‌ها را به فهرست کلمه‌هایی که می‌توان با آن حرف‌ها نوشت نگاشت می‌کرد. برای مثال، کلید <code dir="ltr">'opst'</code> به فهرست <code dir="ltr">['opts', 'post', 'pots', 'spot', 'stop', 'tops']</code> نگاشت می‌شود.

از تابع زیر برای مرتب کردن حرف‌های یک کلمه استفاده می‌کنیم.

</div>

In [ ]:
def sort_word(word):
    return ''.join(sorted(word))

<div dir="rtl" style="text-align: right;">

و این یک مثال است.

</div>

In [ ]:
word = 'pots'
key = sort_word(word)
key

<div dir="rtl" style="text-align: right;">

حالا قفسه‌ای به نام <code dir="ltr">anagram_map</code> باز می‌کنیم. آرگومان <code dir="ltr">'n'</code> یعنی همیشه باید یک قفسه‌ی جدید و خالی بسازیم، حتی اگر از قبل وجود داشته باشد.

</div>

In [ ]:
db = shelve.open('anagram_map', 'n')

<div dir="rtl" style="text-align: right;">

حالا می‌توانیم یک مورد را این‌طور به قفسه اضافه کنیم.

</div>

In [ ]:
db[key] = [word]
db[key]

<div dir="rtl" style="text-align: right;">

در این مورد، کلید رشته و مقدار فهرستی از رشته‌هاست.

حالا فرض کنید کلمه‌ی دیگری پیدا کنیم که همان حرف‌ها را دارد، مثل <code dir="ltr">tops</code>.

</div>

In [ ]:
word = 'tops'
key = sort_word(word)
key

<div dir="rtl" style="text-align: right;">

کلید همان کلید مثال قبلی است، پس می‌خواهیم کلمه‌ی دومی را به همان فهرست رشته‌ها اضافه کنیم. اگر <code dir="ltr">db</code> دیکشنری بود، این‌طور انجامش می‌دادیم.

</div>

In [ ]:
db[key].append(word)          # INCORRECT

<div dir="rtl" style="text-align: right;">

اما اگر این را اجرا کنیم و بعد کلید را در قفسه جست‌وجو کنیم، به نظر می‌رسد به‌روزرسانی نشده است.

</div>

In [ ]:
db[key]

<div dir="rtl" style="text-align: right;">

مشکل این است: وقتی کلید را جست‌وجو می‌کنیم، فهرستی از رشته‌ها می‌گیریم، اما اگر فهرست رشته‌ها را تغییر دهیم، بر قفسه اثری ندارد. اگر بخواهیم قفسه را به‌روزرسانی کنیم، باید مقدار قدیمی را بخوانیم، به‌روزش کنیم و بعد مقدار جدید را دوباره در قفسه بنویسیم.

</div>

In [ ]:
anagram_list = db[key]
anagram_list.append(word)
db[key] = anagram_list

<div dir="rtl" style="text-align: right;">

حالا مقدار قفسه به‌روزرسانی شده است.

</div>

In [ ]:
db[key]

<div dir="rtl" style="text-align: right;">

به‌عنوان تمرین، می‌توانید این مثال را با خواندن فهرست کلمه‌ها و ذخیره‌ی همه‌ی آناگرام‌ها در یک قفسه کامل کنید.

</div>

In [ ]:
db.close()

<div dir="rtl" style="text-align: right;">

## ۱۳.۶. بررسی فایل‌های هم‌ارز

حالا به هدف این فصل برگردیم: جست‌وجوی فایل‌های متفاوتی که داده‌ی یکسانی دارند. یک راه برای بررسی این است که محتویات هر دو فایل را بخوانیم و مقایسه کنیم.

اگر فایل‌ها تصویر داشته باشند، باید آن‌ها را با حالت <code dir="ltr">'rb'</code> باز کنیم، که <code dir="ltr">'r'</code> یعنی می‌خواهیم محتویات را بخوانیم و <code dir="ltr">'b'</code> **حالت دودویی** (binary mode) را نشان می‌دهد. در حالت دودویی، محتویات به‌عنوان متن تفسیر نمی‌شوند – با آن‌ها مثل دنباله‌ای از بایت‌ها رفتار می‌شود.

این مثالی است که یک فایل تصویری را باز می‌کند و می‌خواند.

</div>

In [ ]:
path1 = 'photos/jan-2023/photo1.jpg'
data1 = open(path1, 'rb').read()
type(data1)

<div dir="rtl" style="text-align: right;">

نتیجه‌ی <code dir="ltr">read</code> یک شیء <code dir="ltr">bytes</code> است – همان‌طور که از نامش پیداست، دنباله‌ای از بایت‌ها دارد.

به‌طور کلی محتویات یک فایل تصویری برای انسان خواناست. اما اگر محتویات فایل دومی را بخوانیم، می‌توانیم از عملگر <code dir="ltr">==</code> برای مقایسه استفاده کنیم. (در کتاب اصلی اینجا جمله‌ی «برای انسان خوانا نیست» آمده است؛ منظور این است که محتویات فایل تصویری را نمی‌توان به‌صورت متن خواند.)

</div>

In [ ]:
path2 = 'photos/jan-2023/photo2.jpg'
data2 = open(path2, 'rb').read()
data1 == data2

<div dir="rtl" style="text-align: right;">

این دو فایل هم‌ارز نیستند.

آنچه تا اینجا داریم را در یک تابع کپسوله می‌کنیم.

</div>

In [ ]:
def same_contents(path1, path2):
    data1 = open(path1, 'rb').read()
    data2 = open(path2, 'rb').read()
    return data1 == data2

<div dir="rtl" style="text-align: right;">

اگر فقط دو فایل داشته باشیم، این تابع گزینه‌ی خوبی است. اما فرض کنید تعداد زیادی فایل داریم و می‌خواهیم بدانیم آیا دو تا از آن‌ها داده‌ی یکسانی دارند. ناکارآمد است که هر جفت فایل را مقایسه کنیم.

یک جایگزین استفاده از **تابع درهم‌ساز** (hash function) است که محتویات یک فایل را می‌گیرد و یک **چکیده** (digest) حساب می‌کند، که معمولاً یک عدد صحیح بزرگ است. اگر دو فایل داده‌ی یکسانی داشته باشند، چکیده‌ی یکسانی خواهند داشت. اگر دو فایل فرق داشته باشند، *تقریباً همیشه* چکیده‌های متفاوتی دارند.

ماژول <code dir="ltr">hashlib</code> چند تابع درهم‌ساز فراهم می‌کند – آنی که استفاده می‌کنیم <code dir="ltr">md5</code> نام دارد. با استفاده از <code dir="ltr">hashlib.md5</code> یک شیء <code dir="ltr">HASH</code> می‌سازیم.

</div>

In [ ]:
import hashlib

md5_hash = hashlib.md5()
type(md5_hash)

<div dir="rtl" style="text-align: right;">

شیء <code dir="ltr">HASH</code> متدی به نام <code dir="ltr">update</code> دارد که محتویات فایل را به‌عنوان آرگومان می‌گیرد.

</div>

In [ ]:
md5_hash.update(data1)

<div dir="rtl" style="text-align: right;">

حالا می‌توانیم از <code dir="ltr">hexdigest</code> برای گرفتن چکیده به‌صورت رشته‌ای از ارقام شانزده‌شانزدهی (hexadecimal) استفاده کنیم که یک عدد صحیح را در مبنای ۱۶ نشان می‌دهند.

</div>

In [ ]:
digest = md5_hash.hexdigest()
digest

<div dir="rtl" style="text-align: right;">

تابع زیر این مراحل را کپسوله می‌کند.

</div>

In [ ]:
def md5_digest(filename):
    data = open(filename, 'rb').read()
    md5_hash = hashlib.md5()
    md5_hash.update(data)
    digest = md5_hash.hexdigest()
    return digest

<div dir="rtl" style="text-align: right;">

اگر محتویات یک فایل دیگر را درهم‌سازی کنیم، می‌توانیم تأیید کنیم که چکیده‌ی متفاوتی می‌گیریم.

</div>

In [ ]:
filename2 = 'photos/feb-2023/photo2.jpg'
md5_digest(filename2)

<div dir="rtl" style="text-align: right;">

حالا تقریباً هر چیزی را که برای پیدا کردن فایل‌های هم‌ارز لازم داریم داریم. قدم آخر جست‌وجوی یک دایرکتوری و پیدا کردن همه‌ی فایل‌های تصویری است.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۳.۷. پیمایش دایرکتوری‌ها

تابع زیر دایرکتوری‌ای را که می‌خواهیم جست‌وجو کنیم به‌عنوان آرگومان می‌گیرد. از <code dir="ltr">listdir</code> برای پیمایش محتویات دایرکتوری استفاده می‌کند. وقتی فایلی پیدا کند، مسیر کاملش را چاپ می‌کند. وقتی دایرکتوری‌ای پیدا کند، خودش را به‌صورت بازگشتی برای جست‌وجوی زیردایرکتوری فراخوانی می‌کند.

</div>

In [ ]:
def walk(dirname):
    for name in os.listdir(dirname):
        path = os.path.join(dirname, name)

        if os.path.isfile(path):
            print(path)
        elif os.path.isdir(path):
            walk(path)

<div dir="rtl" style="text-align: right;">

می‌توانیم آن را این‌طور به کار ببریم:

</div>

In [ ]:
walk('photos')

<div dir="rtl" style="text-align: right;">

ترتیب نتیجه‌ها به جزئیات سیستم‌عامل بستگی دارد.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۳.۸. اشکال‌زدایی

وقتی فایل می‌خوانید و می‌نویسید، ممکن است به مشکل فاصله‌ها (whitespace) بخورید. اشکال‌زدایی این خطاها دشوار است چون نویسه‌های فاصله معمولاً نامرئی هستند. برای مثال، این رشته‌ای است که فاصله، یک تب که با دنباله‌ی <code dir="ltr">\t</code> نشان داده شده و یک خط جدید که با دنباله‌ی <code dir="ltr">\n</code> نشان داده شده دارد. وقتی آن را چاپ می‌کنیم، نویسه‌های فاصله را نمی‌بینیم.

</div>

In [ ]:
s = '1 2\t 3\n 4'
print(s)

<div dir="rtl" style="text-align: right;">

تابع داخلی <code dir="ltr">repr</code> می‌تواند کمک کند. هر شیء را به‌عنوان آرگومان می‌گیرد و نمایش رشته‌ای شیء را برمی‌گرداند. برای رشته‌ها، نویسه‌های فاصله را با دنباله‌های بک‌اسلش نشان می‌دهد.

</div>

In [ ]:
print(repr(s))

<div dir="rtl" style="text-align: right;">

این می‌تواند برای اشکال‌زدایی مفید باشد.

مشکل دیگری که ممکن است به آن بخورید این است که سیستم‌های مختلف از نویسه‌های متفاوتی برای نشان دادن پایان یک خط استفاده می‌کنند. بعضی سیستم‌ها خط جدید، که با <code dir="ltr">\n</code> نشان داده می‌شود، به کار می‌برند. بعضی دیگر نویسه‌ی بازگشت (return)، که با <code dir="ltr">\r</code> نشان داده می‌شود. بعضی هر دو را. اگر فایل‌ها را بین سیستم‌های مختلف جابه‌جا کنید، این ناسازگاری‌ها می‌توانند مشکل ایجاد کنند.

بزرگی و کوچکی حروف نام فایل‌ها مسئله‌ی دیگری است که اگر با سیستم‌عامل‌های مختلف کار کنید ممکن است با آن مواجه شوید. در macOS و یونیکس، نام فایل‌ها می‌توانند حروف کوچک و بزرگ، رقم‌ها و بیشتر نمادها را داشته باشند. اما بسیاری از برنامه‌های ویندوز تفاوت حروف کوچک و بزرگ را نادیده می‌گیرند و چند نماد که در macOS و یونیکس مجازند در ویندوز مجاز نیستند.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۳.۹. واژه‌نامه

**زودگذر (ephemeral):** برنامه‌ی زودگذر معمولاً مدت کوتاهی اجرا می‌شود و وقتی تمام می‌شود داده‌هایش از بین می‌رود.

**پایدار (persistent):** برنامه‌ی پایدار نامحدود اجرا می‌شود و دست‌کم بخشی از داده‌هایش را در ذخیره‌سازی دائمی نگه می‌دارد.

**دایرکتوری (directory):** مجموعه‌ای از فایل‌ها و دایرکتوری‌های دیگر.

**دایرکتوری کاری فعلی (current working directory):** دایرکتوری پیش‌فرضی که یک برنامه استفاده می‌کند مگر اینکه دایرکتوری دیگری مشخص شود.

**مسیر (path):** رشته‌ای که دنباله‌ای از دایرکتوری‌ها را مشخص می‌کند، اغلب منتهی به یک فایل.

**مسیر نسبی (relative path):** مسیری که از دایرکتوری کاری فعلی، یا دایرکتوری مشخص‌شده‌ی دیگری، شروع می‌شود.

**مسیر مطلق (absolute path):** مسیری که به دایرکتوری فعلی بستگی ندارد.

**f-string:** رشته‌ای که حرف <code dir="ltr">f</code> قبل از علامت نقل‌قول بازکننده دارد و یک یا چند عبارت در آکولاد دارد.

**داده‌های پیکربندی (configuration data):** داده‌ای، اغلب ذخیره‌شده در یک فایل، که مشخص می‌کند برنامه چه کاری و چگونه باید انجام دهد.

**سریال‌سازی (serialization):** تبدیل یک شیء به رشته.

**سریال‌زدایی (deserialization):** تبدیل یک رشته به شیء.

**پایگاه داده (database):** فایلی که محتویاتش طوری سازمان‌دهی شده تا عملیات خاصی را کارآمد انجام دهد.

**ذخیره‌گاه کلید-مقدار (key-value store):** پایگاه داده‌ای که محتویاتش مثل دیکشنری سازمان‌دهی شده، با کلیدهایی که با مقدارها متناظرند.

**حالت دودویی (binary mode):** شیوه‌ای برای باز کردن فایل که محتویات را به‌جای دنباله‌ای از نویسه‌ها، به‌صورت دنباله‌ای از بایت‌ها تفسیر می‌کند.

**تابع درهم‌ساز (hash function):** تابعی که شیئی را می‌گیرد و عدد صحیحی حساب می‌کند که گاهی چکیده نامیده می‌شود.

**چکیده (digest):** نتیجه‌ی یک تابع درهم‌ساز، به‌ویژه وقتی برای بررسی یکسان بودن دو شیء به کار می‌رود.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۳.۱۰. تمرین‌ها

</div>

In [ ]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

<div dir="rtl" style="text-align: right;">

### ۱۳.۱۰.۱. از یک دستیار مجازی بپرسید

چند موضوع در این فصل مطرح شد که به تفصیل توضیح ندادم. این‌ها پرسش‌هایی هستند که می‌توانید از یک دستیار مجازی بپرسید تا اطلاعات بیشتری بگیرید.

* "What are the differences between ephemeral and persistent programs?" (تفاوت برنامه‌های زودگذر و پایدار چیست؟)

* "What are some examples of persistent programs?" (چند مثال از برنامه‌های پایدار کدام‌اند؟)

* "What's the difference between a relative path and an absolute path?" (تفاوت مسیر نسبی و مسیر مطلق چیست؟)

* "Why does the <code dir="ltr">yaml</code> module have functions called <code dir="ltr">load</code> and <code dir="ltr">safe_load</code>?" (چرا ماژول <code dir="ltr">yaml</code> توابعی به نام <code dir="ltr">load</code> و <code dir="ltr">safe_load</code> دارد؟)

* "When I write a Python shelf, what are the files with suffixes <code dir="ltr">dat</code> and <code dir="ltr">dir</code>?" (وقتی یک قفسه‌ی پایتون می‌نویسم، فایل‌هایی با پسوند <code dir="ltr">dat</code> و <code dir="ltr">dir</code> چیستند؟)

* "Other than key-values stores, what other kinds of databases are there?" (غیر از ذخیره‌گاه‌های کلید-مقدار، چه نوع پایگاه داده‌های دیگری وجود دارد؟)

* "When I read a file, what's the difference between binary mode and text mode?" (وقتی فایلی را می‌خوانم، تفاوت حالت دودویی و حالت متنی چیست؟)

* "What are the differences between a bytes object and a string?" (تفاوت شیء bytes و رشته چیست؟)

* "What is a hash function?" (تابع درهم‌ساز چیست؟)

* "What is an MD5 digest?" (چکیده‌ی MD5 چیست؟)

مثل همیشه، اگر در هر یک از تمرین‌های زیر گیر کردید، از یک دستیار مجازی کمک بگیرید. همراه با پرسش‌تان، ممکن است بخواهید توابع مربوط از این فصل را هم در آن بچسبانید.

</div>

<div dir="rtl" style="text-align: right;">

### ۱۳.۱۰.۲. تمرین

تابعی به نام <code dir="ltr">replace_all</code> بنویسید که به‌عنوان آرگومان یک رشته‌ی الگو، یک رشته‌ی جایگزین و دو نام فایل بگیرد. باید فایل اول را بخواند و محتویاتش را در فایل دوم بنویسد (و اگر لازم است آن را بسازد). اگر رشته‌ی الگو در هر جای محتویات ظاهر شد، باید با رشته‌ی جایگزین عوض شود.

این یک طرح کلی از تابع برای شروع است.

</div>

In [ ]:
def replace_all(old, new, source_path, dest_path):
    # read the contents of the source file
    reader = open(source_path)

    # replace the old string with the new
    
    # write the result into the destination file

In [ ]:
# Solution goes here

<div dir="rtl" style="text-align: right;">

برای آزمودن تابع‌تان، فایل <code dir="ltr">photos/notes.txt</code> را بخوانید، <code dir="ltr">'photos'</code> را با <code dir="ltr">'images'</code> جایگزین کنید و نتیجه را در فایل <code dir="ltr">photos/new_notes.txt</code> بنویسید.

</div>

In [ ]:
source_path = 'photos/notes.txt'
open(source_path).read()

In [ ]:
dest_path = 'photos/new_notes.txt'
old = 'photos'
new = 'images'
replace_all(old, new, source_path, dest_path)

In [ ]:
open(dest_path).read()

<div dir="rtl" style="text-align: right;">

### ۱۳.۱۰.۳. تمرین

در بخش قبل از ماژول <code dir="ltr">shelve</code> برای ساختن یک ذخیره‌گاه کلید-مقدار استفاده کردیم که رشته‌ی مرتب‌شده‌ی حرف‌ها را به فهرستی از آناگرام‌ها نگاشت می‌کند. برای کامل کردن مثال، تابعی به نام <code dir="ltr">add_word</code> بنویسید که یک رشته و یک شیء قفسه را به‌عنوان آرگومان بگیرد.

باید حرف‌های کلمه را مرتب کند تا کلید بسازد، بعد بررسی کند که آیا کلید از قبل در قفسه هست یا نه. اگر نبود، باید فهرستی شامل کلمه‌ی جدید بسازد و به قفسه اضافه کند. اگر بود، باید کلمه‌ی جدید را به مقدار موجود اضافه کند.

</div>

In [ ]:
# Solution goes here

<div dir="rtl" style="text-align: right;">

می‌توانید از این حلقه برای آزمودن تابع‌تان استفاده کنید.

</div>

In [ ]:
download('https://raw.githubusercontent.com/Alireza-Akhavan/ThinkPython/main/words.txt');

In [ ]:
word_list = open('words.txt').read().split()

db = shelve.open('anagram_map', 'n')
for word in word_list:
    add_word(word, db)

<div dir="rtl" style="text-align: right;">

اگر همه‌چیز درست کار کند، باید بتوانید کلیدی مثل <code dir="ltr">'opst'</code> را جست‌وجو کنید و فهرستی از کلمه‌هایی که می‌توان با آن حرف‌ها نوشت بگیرید.

</div>

In [ ]:
db['opst']

In [ ]:
for key, value in db.items():
    if len(value) > 8:
        print(value)

<div dir="rtl" style="text-align: right;">

(ترتیب چاپ این فهرست‌ها به پیاده‌سازی پایگاه داده بستگی دارد.)

</div>

In [ ]:
db.close()

<div dir="rtl" style="text-align: right;">

### ۱۳.۱۰.۴. تمرین

در مجموعه‌ی بزرگی از فایل‌ها، ممکن است بیش از یک نسخه از یک فایل یکسان وجود داشته باشد، که در دایرکتوری‌های مختلف یا با نام‌های مختلف ذخیره شده‌اند. هدف این تمرین جست‌وجوی تکراری‌هاست. به‌عنوان مثال، با فایل‌های تصویری دایرکتوری <code dir="ltr">photos</code> کار می‌کنیم.

این‌طور کار می‌کند:

* از تابع <code dir="ltr">walk</code> بخش ۱۳.۷ برای جست‌وجوی این دایرکتوری برای فایل‌هایی که به یکی از پسوندهای <code dir="ltr">config['extensions']</code> ختم می‌شوند استفاده می‌کنیم.

* برای هر فایل، از <code dir="ltr">md5_digest</code> بخش ۱۳.۶ برای محاسبه‌ی چکیده‌ی محتویات استفاده می‌کنیم.

* با یک قفسه، نگاشتی از هر چکیده به فهرستی از مسیرهایی با آن چکیده می‌سازیم.

* در پایان، قفسه را برای چکیده‌هایی که به چند فایل نگاشت می‌شوند جست‌وجو می‌کنیم.

* اگر پیدا کردیم، از <code dir="ltr">same_contents</code> برای تأیید اینکه فایل‌ها داده‌ی یکسانی دارند استفاده می‌کنیم.

ابتدا چند تابع پیشنهاد می‌کنم که بنویسید، بعد همه را کنار هم می‌گذاریم.

۱. برای شناسایی فایل‌های تصویری، تابعی به نام <code dir="ltr">is_image</code> بنویسید که یک مسیر و فهرستی از پسوندهای فایل را بگیرد و اگر مسیر به یکی از پسوندهای فهرست ختم می‌شود <code dir="ltr">True</code> برگرداند. راهنمایی: از <code dir="ltr">os.path.splitext</code> استفاده کنید – یا از یک دستیار مجازی بخواهید این تابع را برایتان بنویسد.

</div>

In [ ]:
# Solution goes here

<div dir="rtl" style="text-align: right;">

می‌توانید از <code dir="ltr">doctest</code> برای آزمودن تابع‌تان استفاده کنید.

</div>

In [ ]:
from doctest import run_docstring_examples

def run_doctests(func):
    run_docstring_examples(func, globals(), name=func.__name__)

run_doctests(is_image)

<div dir="rtl" style="text-align: right;">

۲. تابعی به نام <code dir="ltr">add_path</code> بنویسید که یک مسیر و یک قفسه را به‌عنوان آرگومان بگیرد. باید از <code dir="ltr">md5_digest</code> برای محاسبه‌ی چکیده‌ی محتویات فایل استفاده کند. بعد باید قفسه را به‌روزرسانی کند: یا موردی جدید بسازد که چکیده را به فهرستی شامل مسیر نگاشت می‌کند، یا اگر مورد وجود دارد مسیر را به فهرست اضافه کند.

</div>

In [ ]:
# Solution goes here

<div dir="rtl" style="text-align: right;">

۳. نسخه‌ای از <code dir="ltr">walk</code> به نام <code dir="ltr">walk_images</code> بنویسید که یک دایرکتوری را بگیرد و فایل‌های دایرکتوری و زیردایرکتوری‌هایش را پیمایش کند. برای هر فایل، باید با <code dir="ltr">is_image</code> بررسی کند که فایل تصویری است یا نه و با <code dir="ltr">add_path</code> آن را به قفسه اضافه کند.

</div>

In [ ]:
# Solution goes here

<div dir="rtl" style="text-align: right;">

وقتی همه‌چیز کار کرد، می‌توانید از برنامه‌ی زیر برای ساختن قفسه، جست‌وجوی دایرکتوری <code dir="ltr">photos</code> و افزودن مسیرها به قفسه، و بعد بررسی اینکه آیا چند فایل با چکیده‌ی یکسان وجود دارد استفاده کنید.

</div>

In [ ]:
db = shelve.open('photos/digests', 'n')
walk_images('photos')

for digest, paths in db.items():
    if len(paths) > 1:
        print(paths)

<div dir="rtl" style="text-align: right;">

باید یک جفت فایل با چکیده‌ی یکسان پیدا کنید. از <code dir="ltr">same_contents</code> برای بررسی اینکه آیا داده‌ی یکسانی دارند استفاده کنید.

</div>

In [ ]:
# Solution goes here

<div dir="rtl" style="text-align: right;">

---

*منبع: Allen B. Downey، «Think Python»، ویرایش سوم، فصل ۱۳. متن اصلی با مجوز [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) و کدها با [مجوز MIT](https://mit-license.org/) منتشر شده‌اند. این ترجمه یک اثر مشتق است و با همان مجوز CC BY-NC-SA 4.0 منتشر می‌شود.*

</div>

<div dir="rtl" style="text-align: right;">

---

**نسخه‌ی وب این جزوه:** فصل ۱۳ کتاب (**فایل‌ها و پایگاه‌های داده**) را در سایت پای‌دان نیز می‌توانید بخوانید: [https://pydaan.ir/translations/think-python/13-files-databases/](https://pydaan.ir/translations/think-python/13-files-databases/)

</div>